# Ropedia Academy — B5 · NeRF & volume rendering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B5.ipynb)

> **Implements NeRF's positional encoding + differentiable volume rendering and plots the density/weights along the ray with the resulting pixel color.**
>
> 实现 NeRF 的位置编码 + 可微体渲染，画出沿射线的密度/权重曲线，以及最终得到的像素颜色。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B5

In [ ]:
import torch, torch.nn as nn, matplotlib.pyplot as plt

# NeRF: an MLP (x) -> (color, density). Positional encoding lets it fit detail.
def posenc(x, L=6):
    out = [x]
    for i in range(L):
        for fn in (torch.sin, torch.cos): out.append(fn(x * (2.0**i) * 3.14159))
    return torch.cat(out, -1)
mlp = nn.Sequential(nn.Linear(3*(1+2*6), 128), nn.ReLU(), nn.Linear(128, 4))
field = lambda p: (torch.sigmoid(mlp(posenc(p))[..., :3]), torch.relu(mlp(posenc(p))[..., 3]))

t = torch.linspace(2, 6, 64)                       # samples along one ray
pts = torch.stack([torch.zeros(64), torch.zeros(64), t], -1)
color, sigma = field(pts)
delta = (t[1]-t[0]).expand(64)
alpha = 1 - torch.exp(-sigma * delta)              # per-sample opacity
Tr = torch.cumprod(torch.cat([torch.ones(1), 1-alpha+1e-10])[:-1], 0)  # transmittance
w = Tr * alpha                                      # sample weights
pixel = (w[:, None] * color).sum(0)                # integrate color along the ray
print("rendered pixel RGB:", pixel.detach().round(decimals=3).tolist())

# Visualize the volume-rendering integral: density, weights, and the output color.
fig, ax = plt.subplots(1, 2, figsize=(7.5, 3.2))
ax[0].plot(t, sigma.detach(), label="density σ"); ax[0].plot(t, w.detach(), label="weight w")
ax[0].set_xlabel("t along ray"); ax[0].legend(); ax[0].set_title("what the ray integrates")
ax[1].imshow(pixel.detach().clamp(0,1).reshape(1,1,3)); ax[1].axis('off'); ax[1].set_title("rendered pixel")
plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B5
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks